In [1]:
import pandas as pd

Here we use the chembl_websource_client to scrape all of the data points related to human liver microsom clearance.

For this we define the keywords:
    "human liver microsom",
    "human liver microsomal",
    "hlm",

and add that they should be CL, meaning related to clearance

In [3]:
from chembl_webresource_client.new_client import new_client
from chembl_webresource_client.settings import Settings
from collections import Counter
import csv
import os

# -----------------------------
# Client settings
# -----------------------------
Settings.Instance().TIMEOUT = 30
Settings.Instance().CACHING = True
Settings.Instance().MAX_LIMIT = 1000

assay = new_client.assay
activity = new_client.activity
molecule = new_client.molecule
target = new_client.target

OUTPUT_DIR = "raw_datasets/chembl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPECIES = "mouse"   # change to "human" for HLM

# -----------------------------
# Helpers
# -----------------------------
def chunks(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

def is_microsome_description(text: str, species: str = "mouse") -> bool:
    if not text:
        return False

    t = text.lower()

    species_terms = {
        "mouse": ["mouse", "murine", "mus musculus", "mlm"],
        "human": ["human", "homo sapiens", "hlm"],
        "rat":   ["rat", "rattus norvegicus", "rlm"],
    }

    return (
        any(term in t for term in species_terms.get(species, [species.lower()]))
        and "liver" in t
        and "microsom" in t   # microsome / microsomal
    )

# -----------------------------
# 1) Find candidate MLM assays
# -----------------------------
candidate_terms = [
    "mouse liver microsom",
    "mouse liver microsomal",
    "mus musculus",
    "mlm",
]

assays = {}

for term in candidate_terms:
    results = assay.filter(description__icontains=term).only([
        "assay_chembl_id",
        "description",
        "document_chembl_id",
        "assay_type",
    ])

    for a in results:
        desc = a.get("description") or ""
        if is_microsome_description(desc, species=SPECIES):
            assays[a["assay_chembl_id"]] = {
                "description": desc,
                "document_chembl_id": a.get("document_chembl_id"),
                "assay_type": a.get("assay_type"),
            }

assay_ids = sorted(assays.keys())
print(f"Found {len(assay_ids)} candidate {SPECIES.upper()} liver microsome assays")

# -----------------------------
# 2) Pull ALL activities first
#    and inspect standard_type
# -----------------------------
rows = []
standard_type_counter = Counter()

for batch in chunks(assay_ids, 100):
    acts = activity.filter(
        assay_chembl_id__in=batch
    ).only([
        "activity_id",
        "assay_chembl_id",
        "molecule_chembl_id",
        "target_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "data_validity_comment",
        "document_chembl_id",
        "document_year",
        "activity_comment",
    ])

    for x in acts:
        mol_id = x.get("molecule_chembl_id")
        std_type = (x.get("standard_type") or "").strip()
        std_value = x.get("standard_value")

        if not mol_id or std_value in (None, ""):
            continue

        standard_type_counter[std_type] += 1

        rows.append({
            "activity_id": x.get("activity_id"),
            "molecule_chembl_id": mol_id,
            "target_chembl_id": x.get("target_chembl_id"),
            "assay_chembl_id": x.get("assay_chembl_id"),
            "document_chembl_id": x.get("document_chembl_id"),
            "data_year": x.get("document_year"),
            "standard_type": std_type,
            "standard_relation": x.get("standard_relation"),
            "standard_value": x.get("standard_value"),
            "standard_units": x.get("standard_units"),
            "data_validity_comment": x.get("data_validity_comment"),
            "activity_comment": x.get("activity_comment"),
            "assay_description": assays[x["assay_chembl_id"]]["description"],
            "assay_type": assays[x["assay_chembl_id"]]["assay_type"],
        })

print(f"Collected {len(rows)} total rows from candidate assays")
print("Top standard_type values:")
for k, v in standard_type_counter.most_common(20):
    print(f"  {k!r}: {v}")

# -----------------------------
# 3) Keep CL / CLINT-like rows
#    (adjust after inspecting counts)
# -----------------------------
allowed_standard_types = {"CL", "CLINT", "Clint"}

filtered_rows = [
    r for r in rows
    if (r["standard_type"] in allowed_standard_types)
]

print(f"Rows after standard_type filter: {len(filtered_rows)}")

# -----------------------------
# 4) Optional cleanup:
#    keep exact values only
# -----------------------------
clean_rows = [
    r for r in filtered_rows
    if r["standard_relation"] in (None, "=")
]

print(f"Exact-value rows: {len(clean_rows)}")

# -----------------------------
# 5) Fetch SMILES
# -----------------------------
requested_molecule_ids = sorted({r["molecule_chembl_id"] for r in clean_rows})
smiles_by_mol = {}
returned_molecule_ids = set()

for batch in chunks(requested_molecule_ids, 100):
    mols = molecule.filter(
        molecule_chembl_id__in=batch
    ).only([
        "molecule_chembl_id",
        "molecule_structures",
    ])

    for m in mols:
        mol_id = m.get("molecule_chembl_id")
        returned_molecule_ids.add(mol_id)

        mol_struct = m.get("molecule_structures") or {}
        smiles_by_mol[mol_id] = mol_struct.get("canonical_smiles")

print(f"Unique molecule IDs requested: {len(requested_molecule_ids)}")
print(f"Molecule records returned: {len(returned_molecule_ids)}")

missing_molecule_ids = set(requested_molecule_ids) - returned_molecule_ids
molecule_ids_with_blank_smiles = {
    mol_id for mol_id in returned_molecule_ids
    if not smiles_by_mol.get(mol_id)
}

print(f"Molecule IDs not returned at all: {len(missing_molecule_ids)}")
print(f"Molecule IDs returned but with blank SMILES: {len(molecule_ids_with_blank_smiles)}")

# -----------------------------
# 6) Fetch target names
# -----------------------------
target_ids = sorted({
    r["target_chembl_id"]
    for r in clean_rows
    if r.get("target_chembl_id")
})

targets_used = {}

for batch in chunks(target_ids, 100):
    target_rows = target.filter(
        target_chembl_id__in=batch
    ).only([
        "target_chembl_id",
        "pref_name",
    ])

    for t in target_rows:
        targets_used[t["target_chembl_id"]] = t.get("pref_name")

print(f"Unique targets used: {len(targets_used)}")

# -----------------------------
# 7) Add structure status
# -----------------------------
for r in clean_rows:
    mol_id = r["molecule_chembl_id"]
    target_id = r.get("target_chembl_id")

    r["smiles"] = smiles_by_mol.get(mol_id)
    r["target_name"] = targets_used.get(target_id)

    if mol_id in missing_molecule_ids:
        r["structure_status"] = "molecule_not_returned"
    elif mol_id in molecule_ids_with_blank_smiles:
        r["structure_status"] = "blank_smiles"
    else:
        r["structure_status"] = "smiles_found"

rows_with_structure = [r for r in clean_rows if r["structure_status"] == "smiles_found"]
rows_without_structure = [r for r in clean_rows if r["structure_status"] != "smiles_found"]

print(f"Rows with structure: {len(rows_with_structure)}")
print(f"Rows without structure: {len(rows_without_structure)}")

# -----------------------------
# 8) Write targets dict file
# -----------------------------
targets_path = os.path.join(OUTPUT_DIR, "chembl_mlm_cl_targets.txt")
with open(targets_path, "w", encoding="utf-8") as f:
    f.write("TARGETS = {\n")
    for target_id in sorted(targets_used):
        pref_name = targets_used[target_id]
        pref_name_escaped = (pref_name or "").replace("\\", "\\\\").replace('"', '\\"')
        f.write(f'    "{target_id}": "{pref_name_escaped}",\n')
    f.write("}\n")

print(f"Wrote {targets_path}")

# -----------------------------
# 9) Write CSVs
# -----------------------------
fieldnames = [
    "activity_id",
    "molecule_chembl_id",
    "smiles",
    "structure_status",
    "target_chembl_id",
    "target_name",
    "assay_chembl_id",
    "document_chembl_id",
    "data_year",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "data_validity_comment",
    "activity_comment",
    "assay_type",
    "assay_description",
]

csv_with_structure = os.path.join(OUTPUT_DIR, "chembl_mlm_cl.csv")
csv_without_structure = os.path.join(OUTPUT_DIR, "chembl_mlm_cl_no_structure.csv")

with open(csv_with_structure, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_with_structure)

with open(csv_without_structure, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_without_structure)

print(f"Wrote {csv_with_structure}")
print(f"Wrote {csv_without_structure}")

Found 4355 candidate MOUSE liver microsome assays
Collected 25685 total rows from candidate assays
Top standard_type values:
  'Stability': 7449
  'CL': 6996
  'T1/2': 6865
  'CLH': 1361
  'Drug metabolism': 958
  'IC50': 573
  'ERH': 248
  'Activity': 187
  'Remaining': 186
  'Remaining (without cofactor)': 186
  'Retention_time': 96
  'Inhibition': 92
  'Drug recovery': 87
  'Ratio': 77
  'QH': 58
  'Fu': 50
  'Drug degradation': 47
  'solubility': 23
  'Km': 23
  'F': 23
Rows after standard_type filter: 6996
Exact-value rows: 6177
Unique molecule IDs requested: 5688
Molecule records returned: 5688
Molecule IDs not returned at all: 0
Molecule IDs returned but with blank SMILES: 0
Unique targets used: 3
Rows with structure: 6177
Rows without structure: 0
Wrote raw_datasets/chembl\chembl_mlm_cl_targets.txt
Wrote raw_datasets/chembl\chembl_mlm_cl.csv
Wrote raw_datasets/chembl\chembl_mlm_cl_no_structure.csv


There are multiple various assays related to HLM clearance, like intrinsic, hepatic etc. so we need to separate them, since we will work with intrinsic clearance

In [5]:
import csv
import os
import re
from collections import defaultdict

INPUT_CSV = "raw_datasets/chembl/chembl_mlm_cl.csv"
OUTPUT_DIR = "raw_datasets/chembl"

# Ordered from most specific to least specific
CATEGORY_PATTERNS = [
    (
        "chembl_mlm_unbound",
        [
            r"\bunbound intrinsic\b",
            r"\bintrinsic unbound\b",
            r"\bclint\s*[,/_ -]?\s*u\b",
            r"\bclu\b",
            r"\bfu[,/_ -]?mic\b",
            r"\bmicrosomal unbound clearance\b",
            r"\bunbound clearance\b",
        ],
    ),
    (
        "chembl_mlm_intrinsic",
        [
            r"\bapparent intrinsic\b",
            r"\bintrinsic apparent\b",
            r"\bapparent clint\b",
            r"\bclint\s*[,/_ -]?\s*app\b",
            r"\bclintapp\b",
            r"\bclint[,/_ -]?\s*app\b",
            r"\bintrinsic clearance\b",
            r"\bclint\b",
            r"\bcl int\b",
            r"\bmicrosomal intrinsic clearance\b",
        ],
    ),
    (
        "chembl_mlm_hepatic",
        [
            r"\bhepatic clearance\b",
            r"\bhepatic intrinsic clearance\b",
            r"\bscaled intrinsic\b",
            r"\bscaled clint\b",
            r"\bivive\b",
            r"\bin vivo clearance\b",
            r"\bml/min/kg\b",
            r"\bl/kg/h\b",
            r"\bper\s*kg\b",
        ],
    ),
]

NEGATIVE_PATTERNS = {
    "chembl_mlm_intrinsic": [
        r"\bunbound intrinsic\b",
        r"\bclint\s*[,/_ -]?\s*u\b",
        r"\bclu\b",
        r"\bfu[,/_ -]?mic\b",
        r"\bmicrosomal unbound clearance\b",
        r"\bhepatic clearance\b",
        r"\bhepatic intrinsic clearance\b",
        r"\bscaled intrinsic\b",
        r"\bscaled clint\b",
        r"\bivive\b",
        r"\bml/min/kg\b",
        r"\bl/kg/h\b",
        r"\bper\s*kg\b",
    ],
}

def normalize_text(*parts):
    text = " | ".join(str(p or "") for p in parts)
    text = text.lower()
    text = text.replace("appearant", "apparent")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def matches_any(text, patterns):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

def classify_row(row):
    text = normalize_text(
        row.get("assay_description"),
        row.get("activity_comment"),
        row.get("target_name"),
        row.get("standard_type"),
        row.get("standard_units"),
    )

    for category, patterns in CATEGORY_PATTERNS:
        if matches_any(text, patterns):
            negs = NEGATIVE_PATTERNS.get(category, [])
            if negs and matches_any(text, negs):
                continue
            return category, text

    # plain "apparent" is too ambiguous: keep in unknown
    if matches_any(text, [
        r"\bapparent clearance\b",
        r"\bapparent cl\b",
        r"\bapparent\b",
    ]):
        return "chembl_mlm_unknown", text

    return "chembl_mlm_unknown", text

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(INPUT_CSV, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

        extra_fields = ["classification_text", "classification_bucket"]
        out_fieldnames = fieldnames + [c for c in extra_fields if c not in fieldnames]

        buckets = defaultdict(list)

        for row in reader:
            bucket, text = classify_row(row)
            row["classification_bucket"] = bucket
            row["classification_text"] = text
            buckets[bucket].append(row)

    expected_buckets = [
        "chembl_mlm_intrinsic",
        "chembl_mlm_unbound",
        "chembl_mlm_hepatic",
        "chembl_mlm_unknown",
    ]

    for bucket in expected_buckets:
        buckets[bucket] = buckets.get(bucket, [])

    for bucket in expected_buckets:
        rows = buckets[bucket]
        out_path = os.path.join(OUTPUT_DIR, f"{bucket}.csv")
        with open(out_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=out_fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"Wrote {out_path} ({len(rows)} rows)")

    summary_path = os.path.join(OUTPUT_DIR, "summary_mlm_clint.txt")
    total = sum(len(v) for v in buckets.values())
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Input file: {INPUT_CSV}\n")
        f.write(f"Total rows processed: {total}\n\n")
        for bucket in expected_buckets:
            f.write(f"{bucket}: {len(buckets[bucket])}\n")

    print(f"Wrote {summary_path}")

if __name__ == "__main__":
    main()

Wrote raw_datasets/chembl\chembl_mlm_intrinsic.csv (4897 rows)
Wrote raw_datasets/chembl\chembl_mlm_unbound.csv (190 rows)
Wrote raw_datasets/chembl\chembl_mlm_hepatic.csv (137 rows)
Wrote raw_datasets/chembl\chembl_mlm_unknown.csv (953 rows)
Wrote raw_datasets/chembl\summary_mlm_clint.txt
